In [1]:
!pip install -q transformers accelerate peft bitsandbytes datasets evaluate rouge_score sacrebleu bert_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 83.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cum

In [2]:
import os
import re
import torch
import pandas as pd
from tqdm.auto import tqdm

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import evaluate

In [3]:
BASE_MODEL_PATH = "/kaggle/input/models/qwen-lm/qwen2.5/transformers/1.5b-instruct/1"

VALID_PATH = "/kaggle/input/datasets/flowboat/nlp-code/test-00000-of-00001 (1).parquet"

SAVED_OUTPUT_DIR = "/kaggle/input/qwen-after-train/qwen2_5_1_5b_qlora_summarization"

# Ưu tiên dùng lora_adapter
ADAPTER_PATH = os.path.join(SAVED_OUTPUT_DIR, "lora_adapter")

# Nếu lora_adapter lỗi, đổi sang checkpoint-1347:
# ADAPTER_PATH = os.path.join(SAVED_OUTPUT_DIR, "checkpoint-1347")

ARTICLE_COL = "article"
SUMMARY_COL = "summary"

MAX_LENGTH = 3072
MAX_NEW_TOKENS = 220

OUTPUT_DIR = "/kaggle/working/evaluate_saved_qwen_lora"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Adapter path:", ADAPTER_PATH)
print("Exists:", os.path.exists(ADAPTER_PATH))
print("Files:", os.listdir(ADAPTER_PATH))

Adapter path: /kaggle/input/qwen-after-train/qwen2_5_1_5b_qlora_summarization/lora_adapter
Exists: True
Files: ['adapter_model.safetensors', 'training_args.bin', 'adapter_config.json', 'README.md', 'tokenizer.json', 'tokenizer_config.json', 'chat_template.jinja']


In [4]:
valid_df = pd.read_parquet(VALID_PATH)

valid_df = valid_df[[ARTICLE_COL, SUMMARY_COL]].dropna()
valid_df[ARTICLE_COL] = valid_df[ARTICLE_COL].astype(str).str.strip()
valid_df[SUMMARY_COL] = valid_df[SUMMARY_COL].astype(str).str.strip()

valid_df = valid_df[
    (valid_df[ARTICLE_COL] != "") &
    (valid_df[SUMMARY_COL] != "")
].reset_index(drop=True)

print(valid_df.shape)
valid_df.head()

(1344, 2)


,article,summary
0,Văn phòng mới của MayTrip tại địa chỉ 833 Lê H...,MayTrip khai trương văn phòng mới tại TP. HCM ...
1,"Đầu tháng 11, dã quỳ bung nở trên các vạt núi ...",Rừng hoa dã quỳ ở Vườn quốc gia Ba Vì rộng kho...
2,"Mùa thu đông năm nay, Tam Cốc đặc biệt và hấp ...","Mùa thu đông năm nay, Tam Cốc đặc biệt và hấp ..."
3,"Cầm cốc chocolate đen nóng trên tay, nữ du khá...",Du khách Anh Monisha Rajesh bắt đầu hành trình...
4,"Cồn Én nằm giữa sông Tiền, thuộc xã Tấn Mỹ, rộ...",Cồn Én là một điểm đến du lịch nằm giữa sông T...


In [5]:
try:
    tokenizer = AutoTokenizer.from_pretrained(
        ADAPTER_PATH,
        trust_remote_code=True
    )
    print("Loaded tokenizer from adapter.")
except Exception as e:
    print("Cannot load tokenizer from adapter, loading from base model.")
    print(e)

    tokenizer = AutoTokenizer.from_pretrained(
        BASE_MODEL_PATH,
        trust_remote_code=True
    )

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

Loaded tokenizer from adapter.


In [6]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_PATH,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

base_model.config.use_cache = True

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [7]:
model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH
)

model.eval()

print("Loaded LoRA adapter from:", ADAPTER_PATH)

/usr/local/lib/python3.12/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['lora_ga_config', 'use_bdlora'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


Loaded LoRA adapter from: /kaggle/input/qwen-after-train/qwen2_5_1_5b_qlora_summarization/lora_adapter


In [8]:
def build_prompt(article):
    messages = [
        {
            "role": "system",
            "content": "Bạn là trợ lý AI chuyên tóm tắt văn bản tiếng Việt một cách ngắn gọn, đầy đủ ý chính và không thêm thông tin ngoài văn bản."
        },
        {
            "role": "user",
            "content": f"""Hãy tóm tắt văn bản sau bằng tiếng Việt.

Yêu cầu:
- Tóm tắt ngắn gọn
- Giữ các ý chính quan trọng
- Không bịa thêm thông tin
- Viết thành một đoạn văn mạch lạc

Văn bản:
{article}

Tóm tắt:"""
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    return prompt

In [9]:
def summarize_after_finetune(article, max_new_tokens=MAX_NEW_TOKENS):
    prompt = build_prompt(article)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_ids = output_ids[0][inputs["input_ids"].shape[-1]:]
    text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    return text.strip()

In [10]:
idx = 0

print("ARTICLE:")
print(valid_df.loc[idx, ARTICLE_COL][:1000])

print("\nREFERENCE:")
print(valid_df.loc[idx, SUMMARY_COL])

print("\nPRED:")
print(summarize_after_finetune(valid_df.loc[idx, ARTICLE_COL]))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


ARTICLE:
Văn phòng mới của MayTrip tại địa chỉ 833 Lê Hồng Phong, quận 10, nhằm mở rộng dịch vụ và nâng cao trải nghiệm khách hàng khu vực phía Nam. Văn phòng mới được kỳ vọng là điểm đến thuận tiện cho khách hàng trong việc tìm hiểu và sử dụng dịch vụ của công ty. Trong ngày khai trương, MayTrip trao tặng nhiều quà tặng và ưu đãi cho khách tham dự. Các voucher gồm: 2 triệu đồng cho khách đăng ký tour Bắc Á trong ba ngày khai trương, 1 triệu đồng cho tour Đông Nam Á trong ba tháng từ ngày khai trương, 500.000 đồng cho dịch vụ tour nội địa trong một năm, 200.000 đồng dịch vụ đặt phòng khách sạn hoặc xe du lịch trong 6 tháng và 100.000 đồng với dịch vụ vé máy bay trong 6 tháng. Buổi lễ khai trương văn phòng tại Sài Gòn còn giới thiệu các chương trìnhtour Nhật Bảndịp Tết Âm lịch như cung đường vàng trải nghiệm trượt tuyết, Hokkaido - miền tuyết trắng. Những chuyến đi này được MayTrip thiết kế kỹ lưỡng, phù hợp với không khí đón xuân cho gia đình và cặp đôi Việt trong dịp Tết Nguyên đán. N

In [11]:
predictions = []
references = []
articles = []

num_valid_samples = len(valid_df)

for i in tqdm(range(num_valid_samples), desc="Generate full valid"):
    article = valid_df.iloc[i][ARTICLE_COL]
    reference = valid_df.iloc[i][SUMMARY_COL]

    pred = summarize_after_finetune(article)

    articles.append(article)
    references.append(reference)
    predictions.append(pred)

pred_df = pd.DataFrame({
    "article": articles,
    "reference_summary": references,
    "predicted_summary": predictions,
})

PRED_PATH = os.path.join(OUTPUT_DIR, "valid_predictions_from_saved_lora_full.csv")
pred_df.to_csv(PRED_PATH, index=False, encoding="utf-8-sig")

print("Saved predictions to:", PRED_PATH)
pred_df.head()

Generate full valid:   0%|          | 0/1344 [00:00<?, ?it/s]

Saved predictions to: /kaggle/working/evaluate_saved_qwen_lora/valid_predictions_from_saved_lora_full.csv


,article,reference_summary,predicted_summary
0,Văn phòng mới của MayTrip tại địa chỉ 833 Lê H...,MayTrip khai trương văn phòng mới tại TP. HCM ...,MayTrip đã khai trương văn phòng mới tại địa c...
1,"Đầu tháng 11, dã quỳ bung nở trên các vạt núi ...",Rừng hoa dã quỳ ở Vườn quốc gia Ba Vì rộng kho...,Dã quỳ đang bắt đầu bung nở trên các vạt núi v...
2,"Mùa thu đông năm nay, Tam Cốc đặc biệt và hấp ...","Mùa thu đông năm nay, Tam Cốc đặc biệt và hấp ...","Mùa thu đông năm nay, Tam Cốc trở nên hấp dẫn ..."
3,"Cầm cốc chocolate đen nóng trên tay, nữ du khá...",Du khách Anh Monisha Rajesh bắt đầu hành trình...,Nữ du khách Anh Monisha Rajesh đã trải qua chu...
4,"Cồn Én nằm giữa sông Tiền, thuộc xã Tấn Mỹ, rộ...",Cồn Én là một điểm đến du lịch nằm giữa sông T...,Khu du lịch sinh thái Cồn Én nằm giữa sông Tiề...


In [12]:
rouge = evaluate.load("rouge")

rouge_scores = rouge.compute(
    predictions=predictions,
    references=references,
)

rouge_scores

{'rouge1': np.float64(0.7531969386705676),
 'rouge2': np.float64(0.4967489488278991),
 'rougeL': np.float64(0.513381228765021),
 'rougeLsum': np.float64(0.5149554347109457)}

In [13]:
sacrebleu = evaluate.load("sacrebleu")

bleu_scores = sacrebleu.compute(
    predictions=predictions,
    references=[[ref] for ref in references],
)

bleu_scores

{'score': 36.59856478518907,
 'counts': [88919, 60738, 47011, 38471],
 'totals': [154774, 153430, 152086, 150742],
 'precisions': [57.45086384017988,
  39.586782245975364,
  30.910800468156175,
  25.521089013015615],
 'bp': 1.0,
 'sys_len': 154774,
 'ref_len': 153611}

In [14]:
if torch.cuda.is_available():
    torch.cuda.empty_cache()

bertscore = evaluate.load("bertscore")
bertscore_device = "cuda" if torch.cuda.is_available() else "cpu"

bertscore_scores = bertscore.compute(
    predictions=predictions,
    references=references,
    model_type="bert-base-multilingual-cased",
    device=bertscore_device,
    batch_size=16,
    rescale_with_baseline=False,
)

bertscore_scores.keys()

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


dict_keys(['precision', 'recall', 'f1', 'hashcode'])

In [15]:
def mean_score(values):
    return float(sum(values) / len(values)) if values else None

summary_metrics = {
    "num_samples": len(predictions),
    "rouge1": float(rouge_scores["rouge1"]),
    "rouge2": float(rouge_scores["rouge2"]),
    "rougeL": float(rouge_scores["rougeL"]),
    "bleu": float(bleu_scores["score"]),
    "bleu_brevity_penalty": float(bleu_scores["bp"]),
    "bleu_length_ratio": float(bleu_scores["sys_len"] / bleu_scores["ref_len"]) if bleu_scores["ref_len"] != 0 else None,
    "bertscore_precision": mean_score(bertscore_scores["precision"]),
    "bertscore_recall": mean_score(bertscore_scores["recall"]),
    "bertscore_f1": mean_score(bertscore_scores["f1"]),
}

summary_df = pd.DataFrame([summary_metrics])

SUMMARY_METRICS_PATH = os.path.join(OUTPUT_DIR, "benchmark_summary_metrics.csv")
summary_df.to_csv(SUMMARY_METRICS_PATH, index=False)

print("Saved benchmark summary to:", SUMMARY_METRICS_PATH)
summary_df

Saved benchmark summary to: /kaggle/working/evaluate_saved_qwen_lora/benchmark_summary_metrics.csv


,num_samples,rouge1,rouge2,rougeL,bleu,bleu_brevity_penalty,bleu_length_ratio,bertscore_precision,bertscore_recall,bertscore_f1
0,1344,0.753197,0.496749,0.513381,36.598565,1.0,1.007571,0.810145,0.808151,0.808853
